# Interactive Prototype

Goal: Create a functional MVP for field route planning and management.

Stages:
1. Region Input & Sub-Division
2. Automatic Target Search & Routing
3. Manual Adjustment
4. Select & Execute Plans

## Region Input & Sub-Division

- Get region outline
- Divide into work cells
- Tentative plan for depot locations
- 

### Dummy Region Shapefile

At this point, we do not actually have a shapefile of the target region. The following two Jupyter cells will generate one using the outline of the orthophoto we have created. 

Load this as the "shapefile" which will define our working region. 

In [1]:
region_image_path = '../input/IGNORE_Brewster-2024-all-orthophoto-UTM-32613.tif'
region_contour_shapefile = '../input/interactive_proto/region_contour.shp'
region_contour_geojson = '../input/interactive_proto/region_contour.geojson'


region_crs = 32613 # Use this everywhere for consistency
visualization_crs = 4326 # Use this when we need leaflet visualizations
simplification_tolerance = 5


In [2]:
import sys
import geopandas as gpd
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

### Create Voronoi Partitioning, Solve for Depots

- [x] Display region outline
- [x] Display region partition cells, centroids
- [ ] Find and indicate depot locations 

In [3]:
target_area_acres = 0.5
# target_area_acres = 1.5
# target_area_acres = 2.5

target_area_sqm = target_area_acres * 4046.86
max_iterations = 15 # Cycles to find improved partition

voronoi_partition_filename = '../input/interactive_proto/voronoi_partition.geojson'
voronoi_centroids_filename = '../input/interactive_proto/voronoi_centroids.geojson'

# Depot placement parameters
depot_radius = 225  # Max distance a depot can cover
depots_filename = '../input/interactive_proto/depot_points.geojson'

In [4]:
from plant_search.region_partition import centroidal_voronoi_tessellation
from macro_planning.depot_placement import find_depots, assign_cells_to_depot

region_outline_gdf = gpd.read_file(region_contour_shapefile)
simplified_polygon = region_outline_gdf.geometry.iloc[0]
# print(loaded_gdf.crs)
num_cells = int(simplified_polygon.area / target_area_sqm) # How many cells to generate

# Divide region into voronoi cells
cell_gdf = centroidal_voronoi_tessellation(simplified_polygon, num_cells, max_iterations)

# Find depots to cover all cells
grid_density = 4
depots_gdf = find_depots(depot_radius, cell_gdf, region_outline_gdf, grid_density)

# for depot_id, depot in depots_gdf.iterrows():
#     print(f'{depot_id}: {depot["geometry"]}')

cell_gdf = assign_cells_to_depot(depots_gdf, cell_gdf)


# for depot_id, depot in updated_cell_gdf.iterrows():
#     print(f'{depot_id}: {depot["closest_depot"]}')

Reached maximum iterations without full convergence.


#### Write Data to Files

1. Region cells
2.  Region cell centroids
3. Depot locations

In [5]:
cell_gdf_4326 = cell_gdf.copy().to_crs(visualization_crs)
# print(cell_gdf_4326.crs)

# Create a copy with only the 'geometry' column (Voronoi polygons)
voronoi_gdf = cell_gdf_4326.copy().drop(columns=["cell_centroid"])
voronoi_gdf.to_crs(visualization_crs, inplace=True)
voronoi_gdf.to_file(voronoi_partition_filename, driver="GeoJSON")

# Create a copy with only the 'cell_centroid' column and set it as the active geometry
centroid_gdf = cell_gdf_4326.copy().drop(columns=["geometry"])
centroid_gdf.set_geometry("cell_centroid", inplace=True)

centroid_gdf.set_crs(region_crs, inplace=True)  # Reset the CRS explicitly
centroid_gdf.to_crs(visualization_crs, inplace=True)  # Reset the CRS explicitly
centroid_gdf.to_file(voronoi_centroids_filename, driver="GeoJSON")

# Write Depot locations to file
depots_gdf.to_crs(visualization_crs, inplace=True)
depots_gdf.to_file(depots_filename, driver="GeoJSON")


#### Data Interaction with Leaflet

In [6]:
from ipyleaflet import Circle, CircleMarker, LayerGroup

def create_depot_layers(depot_data):
    depot_layers = []

    for feature in depot_data["features"]:
        depot_plots = [] # Hold range, centerpoint circles
        coords = feature["geometry"]["coordinates"]
        properties = feature["properties"]
        depot_radius = properties.get("depot_radius", 0)  # Default to 0 if missing
        depot_id = properties.get("depot_id", "Unknown ID")
        depot_name = f'Depot {depot_id}'

        range_circle = Circle(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=depot_radius,  # Circle radius in meters
            color='black', fill=False, fill_color='#3366cc',
            fill_opacity=0.05, weight=1,
            tooltip=f"Depot ID: {depot_id}\nRadius: {depot_radius}m"
        )
        
        center_circle = CircleMarker(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=5,  # Circle radius in meters
            color='black', fill=True, fill_color='red',
            fill_opacity=0.9, weight=1,
            tooltip=f"Depot ID: {depot_id}\nRadius: {depot_radius}m"
        )

        depot_layergroup = LayerGroup(
            layers=(range_circle, center_circle),
            name=depot_name
        )
        depot_layers.append(depot_layergroup)
    
    return depot_layers

In [7]:
import geopandas as gpd
from ipyleaflet import (
    Map, GeoJSON, GeoData, Circle, LayerGroup,
    LayersControl, ScaleControl
)
from shapely.geometry import mapping, shape
import json

# Load the GeoJSON region outline
with open(region_contour_geojson, "r") as f:
    region_contour_data = json.load(f)
region_geometry = shape(region_contour_data['features'][0]['geometry'])
region_center = region_geometry.centroid

# Load Voronoi polygons
with open(voronoi_partition_filename, "r") as f:
    voronoi_data = json.load(f)

# Load centroids
with open(voronoi_centroids_filename, "r") as f:
    centroid_data = json.load(f)

# Load depot locations
with open(depots_filename, "r") as f:
    depot_data = json.load(f)

# print(region_contour_data)
print(voronoi_data)
# print(centroid_data)
# print(depot_data)




m = Map(center=(region_center.y, region_center.x), zoom=16)

# Add the region border to the map
region_layer = GeoJSON(
    data=region_contour_data, 
    style={'color': 'green', 'fillOpacity': 0.2, 'weight': 3},
    name=region_contour_data['name'])
m.add_layer(region_layer)

# Add Voronoi polygons
voronoi_layer = GeoJSON(
    data=voronoi_data, 
    style={'color': 'blue', 'fillColor': 'lightblue', 'opacity': 0.5, 'weight': 2},
    name=voronoi_data['name'])
m.add_layer(voronoi_layer)

# Add centroids
centroid_layer = GeoJSON(
    data=centroid_data, 
    style={'color': 'black', 'radius':3, 'fillColor': '#3366cc', 'opacity':0.5, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': 'red' , 'fillOpacity': 0.2},
    point_style={'radius': 3, 'color': 'red', 'fillOpacity': 0.8, 'fillColor': 'blue', 'weight': 3},
    name=centroid_data['name'])
centroid_layer.visible = False  # Set layer to hidden
m.add_layer(centroid_layer)

# Plot depot circles
depot_layers = create_depot_layers(depot_data)
for depot_layer in depot_layers:
    m.add(depot_layer)


m.add_control(LayersControl(position='topright'))
m.add(ScaleControl(position='bottomleft'))
m # Display the map

{'type': 'FeatureCollection', 'name': 'voronoi_partition', 'crs': {'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}}, 'features': [{'type': 'Feature', 'properties': {'cell_id': 0, 'associated_depots': ['depot_22'], 'closest_depot': 'depot_22'}, 'geometry': {'type': 'Polygon', 'coordinates': [[[-103.60479276220966, 30.24993533490594], [-103.60494787057112, 30.249565808659806], [-103.60544662795903, 30.249698274273513], [-103.60537224400474, 30.249865920578127], [-103.60500726613041, 30.250152651103235], [-103.60479276220966, 30.24993533490594]]]}}, {'type': 'Feature', 'properties': {'cell_id': 1, 'associated_depots': ['depot_22'], 'closest_depot': 'depot_22'}, 'geometry': {'type': 'Polygon', 'coordinates': [[[-103.60494787057112, 30.249565808659806], [-103.60489520999191, 30.24948258098971], [-103.60498430149013, 30.249260592741194], [-103.6054094302913, 30.249099548044292], [-103.60546976273096, 30.249221478768387], [-103.60550211222898, 30.24957322359244], [-103

Map(center=[30.24893165719689, -103.6019209136208], controls=(ZoomControl(options=['position', 'zoom_in_text',…

## Show Routes from each Depot

- Show cells associated with each depot
- Show routes to cover all cells from each depot

In [ ]:
cell_group_gdfs = [x for _, x in cell_gdf.groupby('closest_depot')]

print(len(cell_group_gdfs))
print(type(cell_group_gdfs[0]))

# cell_gdf_dict = cell_gdf.groupby('closest_depot').apply().to_dict()
dict_of_groups = {
    key: group
    for key, group in cell_gdf.groupby('closest_depot')
}

# print(dict_of_groups)

key1 = list(dict_of_groups.keys())[0]
print(len(dict_of_groups.keys()))
print(dict_of_groups[key1])
# print(type(dict_of_groups[key1][0]))

3
<class 'geopandas.geodataframe.GeoDataFrame'>
3
                                             geometry  \
35  POLYGON ((634540.073 3347030.283, 634568.574 3...   
36  POLYGON ((634366.476 3347091.112, 634392.877 3...   
41  POLYGON ((634414.233 3347160.612, 634446.328 3...   
42  POLYGON ((634392.877 3347105.873, 634395.279 3...   
43  POLYGON ((634402.843 3347185.788, 634428.049 3...   
44  POLYGON ((634516.653 3347337.252, 634514.944 3...   
47  POLYGON ((634489.479 3347047.74, 634516.84 334...   
48  POLYGON ((634412.63 3347063.292, 634437.219 33...   
49  POLYGON ((634450.266 3347071.266, 634466.991 3...   
50  POLYGON ((634458.522 3347325.545, 634487.481 3...   
51  POLYGON ((634445.407 3347271.834, 634484.037 3...   
52  POLYGON ((634411.19 3347243.26, 634442.6 33472...   
53  POLYGON ((634460.424 3347131.649, 634480.03 33...   
54  POLYGON ((634448.299 3347206.381, 634464.007 3...   
55  POLYGON ((634458.522 3347325.545, 634458.303 3...   
56  POLYGON ((634446.328 3347160.074, 

KeyError: 0

In [9]:
from ipyleaflet import Choropleth, GeoJSON
import matplotlib as plt

colors = plt.cm.tab20(range(len(depots_gdf)))  # Use tab20 colormap for up to 20 depots
colors = ['red', 'yellow', 'orange', 'green', 'blue', 'purple']
depot_colors = {depot['depot_id']: colors[i] for i, depot in depots_gdf.iterrows()}
print(depot_colors)

cell_coloring = dict(zip(cell_gdf['cell_id'], cell_gdf['closest_depot']))

print(cell_coloring)

feature = voronoi_data['features'][0]
print(feature['properties']['closest_depot'])

def color_cells(feature):
    return {
        'fillColor': depot_colors[str(feature['properties']['closest_depot'])],
        'color': depot_colors[feature['properties']['closest_depot']],
        'opacity': 0.99,
        'weight': 2,
    }

m2 = Map(center=(region_center.y, region_center.x), zoom=16)

voronoi_layer = GeoJSON(
    data=voronoi_data, 
    # style={'color': 'red', 'fillColor': 'lightblue', 'opacity': 0.5, 'weight': 2},
    style_callback=color_cells,
    name=voronoi_data['name'])
m2.add_layer(voronoi_layer)

m2

{'depot_22': 'red', 'depot_145': 'yellow', 'depot_169': 'orange'}
{0: 'depot_22', 1: 'depot_22', 2: 'depot_22', 3: 'depot_22', 4: 'depot_22', 5: 'depot_22', 6: 'depot_22', 7: 'depot_22', 8: 'depot_22', 9: 'depot_22', 10: 'depot_22', 11: 'depot_22', 12: 'depot_22', 13: 'depot_22', 14: 'depot_22', 15: 'depot_22', 16: 'depot_22', 17: 'depot_22', 18: 'depot_22', 19: 'depot_22', 20: 'depot_22', 21: 'depot_22', 22: 'depot_22', 23: 'depot_22', 24: 'depot_22', 25: 'depot_22', 26: 'depot_22', 27: 'depot_22', 28: 'depot_22', 29: 'depot_22', 30: 'depot_22', 31: 'depot_22', 32: 'depot_22', 33: 'depot_22', 34: 'depot_22', 35: 'depot_145', 36: 'depot_145', 37: 'depot_22', 38: 'depot_22', 39: 'depot_22', 40: 'depot_22', 41: 'depot_145', 42: 'depot_145', 43: 'depot_145', 44: 'depot_145', 45: 'depot_22', 46: 'depot_22', 47: 'depot_145', 48: 'depot_145', 49: 'depot_145', 50: 'depot_145', 51: 'depot_145', 52: 'depot_145', 53: 'depot_145', 54: 'depot_145', 55: 'depot_145', 56: 'depot_145', 57: 'depot_145'

Map(center=[30.24893165719689, -103.6019209136208], controls=(ZoomControl(options=['position', 'zoom_in_text',…

In [77]:


def depot_selection_layers(depot_data, cell_data):
    depot_layers = {} # dict, where key is depot_id

    for feature in depot_data["features"]:
        depot_plots = [] # Hold range, centerpoint circles
        coords = feature["geometry"]["coordinates"]
        properties = feature["properties"]
        depot_radius = properties.get("depot_radius", 0)  # Default to 0 if missing
        depot_id = properties.get("depot_id", "Unknown ID")
        depot_name = f'Depot {depot_id}'

        # Find cells associated with each depot for coloration
        associated_cells = cell_data.copy()
        associated_cells['features'] = [feature for feature in cell_data['features']
                                        if feature['properties']['closest_depot'] == depot_id]

        # Add highlight to cells in depot range
        cells_layer = GeoJSON(
            data=associated_cells, 
            style={'color': 'red', 'fillColor': 'lightblue', 'opacity': 0.5, 'weight': 2},
            # style_callback=color_cells,
            name=associated_cells['name'])


        range_circle = Circle(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=depot_radius,  # Circle radius in meters
            color='black', fill=False, fill_color='#3366cc',
            fill_opacity=0.05, weight=1,
            tooltip=f"Depot ID: {depot_id}\nRadius: {depot_radius}m"
        )
        
        center_circle = CircleMarker(
            location=[coords[1], coords[0]],  # GeoJSON uses (lon, lat), Folium expects (lat, lon)
            radius=5,  # Circle radius in meters
            color='black', fill=True, fill_color='red',
            fill_opacity=0.9, weight=1,
            tooltip=f"Depot ID: {depot_id}\nRadius: {depot_radius}m"
        )

        depot_layergroup = LayerGroup(
            layers=(cells_layer, range_circle, center_circle),
            name=depot_name
        )
        depot_layers[depot_id] = depot_layergroup
    
    return depot_layers

In [ ]:
from ipyleaflet import Choropleth, GeoJSON, WidgetControl
from ipywidgets import Select, Dropdown
import matplotlib as plt
from shapely.geometry import mapping, shape
import json


# Load Data for mapping
# =====================

# Load the GeoJSON region outline
with open(region_contour_geojson, "r") as f:
    region_contour_data = json.load(f)
region_geometry = shape(region_contour_data['features'][0]['geometry'])
region_center = region_geometry.centroid

# Load Voronoi cells
with open(voronoi_partition_filename, "r") as f:
    voronoi_data = json.load(f)

# Load depot locations
with open(depots_filename, "r") as f:
    depot_data = json.load(f)



# Set up interactive layer selections
# ===================================
all_depot_layers = depot_selection_layers(depot_data, voronoi_data) # Dict of layer instances
list_depots = list(all_depot_layers.keys())

# Depot select widget
depot_select = Dropdown(
    options=list_depots,
    value=list_depots[0],
    description='Depot:',
    disabled=False
)

def on_depot_select(change):
    old_layer = all_depot_layers[change['old']]
    new_layer = all_depot_layers[change['new']]
    m3.substitute(old_layer, new_layer)

depot_select.observe(on_depot_select, names='value')




# Set up interactive map
# ======================
m3 = Map(center=(region_center.y, region_center.x), zoom=16)

# Add the region border to the map
region_layer = GeoJSON(
    data=region_contour_data, 
    style={'color': 'blue', 'fillOpacity': 0.05, 'weight': 2},
    name=region_contour_data['name'])
m3.add_layer(region_layer)

# Add Voronoi polygons
voronoi_layer = GeoJSON(
    data=voronoi_data, 
    style={'color': 'blue', 'fillColor': 'lightblue', 'opacity': 0.25, 'weight': 1},
    name=voronoi_data['name'])
m3.add(voronoi_layer)

# Always keep depot points visible
depot_points = GeoJSON(
    data=depot_data,
    style={'color': 'black', 'radius':3, 'fillColor': '#3366cc', 'opacity':0.5, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': 'red' , 'fillOpacity': 0.2},
    point_style={'radius': 3, 'color': 'red', 'fillOpacity': 0.8, 'fillColor': 'blue', 'weight': 3},
    name=depot_data['name']
)
m3.add(depot_points)

# Add depot selection dropdown widget
depot_select_control = WidgetControl(widget=depot_select, position='bottomright')
m3.add(depot_select_control)

# Add (interactive + dynamic) depot layer
depot_layer = all_depot_layers[depot_select.value] # Whichever is initially set
m3.add(depot_layer)

m3.add_control(LayersControl(position='topright'))
m3.add(ScaleControl(position='bottomleft'))
m3

Map(center=[30.24893165719689, -103.6019209136208], controls=(ZoomControl(options=['position', 'zoom_in_text',…

In [26]:
depot_data

list_depots = [feature['properties']['depot_id'] for feature in depot_data['features']]
print(list_depots)

['depot_22', 'depot_145', 'depot_169']
